# 05 — Features : vélocité par carte
Chaque feature est calculée uniquement avec les transactions **strictement antérieures**
de la même carte (preuve : `tests/test_velocity.py::test_no_future_leakage`).

y

In [ ]:
import sys
import time
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)

PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SILVER = ROOT / "data" / "processed" / "silver"
TX = SILVER / "transactions.parquet"
FEATURES = ROOT / "data" / "processed" / "features"
VEL = FEATURES / "velocity.parquet"
TMP = ROOT / "data" / "processed" / "tmp"
TMP.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET temp_directory = '{TMP.as_posix()}'")
con.execute("SET memory_limit = '3GB'")           # laisse ~4,5 Gio a VS Code, Jupyter et Ubuntu
con.execute("SET threads = 4")                    # moins de threads = moins de memoire simultanee
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")    # evite l'erreur d'affichage widget-view+json

print("Dossier :", Path.cwd())
print("TX existe :", TX.exists(), "| VEL existe :", VEL.exists())
print(con.sql("""
    SELECT current_setting('memory_limit') AS memoire,
           current_setting('threads')      AS threads,
           current_setting('temp_directory') AS temp
""").df().to_string(index=False))

In [ ]:
from secumon.ml.features import velocity

t0 = time.perf_counter()
n = velocity.build_velocity(con, TX, VEL)
print(f"{n} lignes ecrites en {time.perf_counter() - t0:.0f} s")
print(f"Taille : {VEL.stat().st_size / 1e6:.1f} Mo")

In [ ]:
display(con.sql(f"""
    SELECT COUNT(*)                                                  AS lignes,
           COUNT(*) FILTER (WHERE secs_since_prev IS NULL)           AS sans_precedente,
           COUNT(*) FILTER (WHERE n_tx_1h > n_tx_24h
                               OR n_tx_24h > n_tx_7d)                AS fenetres_incoherentes,
           COUNT(*) FILTER (WHERE amt_sum_1h > amt_sum_24h + 0.001)  AS montants_incoherents,
           COUNT(*) FILTER (WHERE secs_since_prev < 0)               AS ecart_negatif
    FROM '{VEL}'
""").df())

con.sql(f"""
    SELECT MAX(n_tx_1h) AS max_1h, MAX(n_tx_24h) AS max_24h, MAX(n_tx_7d) AS max_7d,
           MAX(n_err_24h) AS max_err_24h, ROUND(MAX(amt_sum_24h), 2) AS max_montant_24h
    FROM '{VEL}'
""").df()

In [ ]:
con.sql(f"""
    SELECT t.is_fraud,
           COUNT(*)                                AS transactions,
           ROUND(AVG(v.n_tx_1h), 3)                AS moy_n_tx_1h,
           ROUND(AVG(v.n_tx_24h), 2)               AS moy_n_tx_24h,
           ROUND(AVG(v.amt_sum_24h), 2)            AS moy_montant_24h,
           ROUND(AVG(v.n_err_24h), 3)              AS moy_err_24h,
           ROUND(MEDIAN(v.secs_since_prev) / 60)   AS mediane_minutes_depuis_prec
    FROM '{TX}' t
    JOIN '{VEL}' v USING (tx_id)
    WHERE t.split = 'train'
    GROUP BY t.is_fraud
    ORDER BY t.is_fraud
""").df()

## Features contextuelles
Montant relatif (90 j, par carte), distance domicile → commerçant, distance et vitesse depuis
l'achat physique précédent (par client), premier pays / MCC / commerçant vu.

In [ ]:
TMP_LINUX = Path.home() / ".cache" / "secumon-duckdb-tmp"
TMP_LINUX.mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory = '{TMP_LINUX.as_posix()}'")
print(con.sql("SELECT current_setting('temp_directory') AS temp").df()["temp"].iloc[0])

In [ ]:
from secumon.ml.features import context

TX_GEO = SILVER / "tx_geo.parquet"
CUSTOMERS = SILVER / "customers.parquet"
CTX = FEATURES / "context.parquet"

t0 = time.perf_counter()
n = context.build_context(con, TX, TX_GEO, CUSTOMERS, CTX)
print(f"{n} lignes ecrites en {time.perf_counter() - t0:.0f} s")
print(f"Taille : {CTX.stat().st_size / 1e6:.1f} Mo")

In [ ]:
TX_GEO = SILVER / "tx_geo.parquet"
CTX = FEATURES / "context.parquet"

In [ ]:
display(con.sql(f"""
    SELECT COUNT(*)                                                    AS lignes,
           COUNT(*) FILTER (WHERE dist_home_km IS NULL)                AS dist_home_vide,
           COUNT(*) FILTER (WHERE country_new IS NULL)                 AS country_new_vide,
           COUNT(*) FILTER (WHERE dist_prev_km IS NULL)                AS dist_prev_vide,
           COUNT(*) FILTER (WHERE dist_home_km > 20016
                               OR dist_prev_km > 20016)                AS distance_impossible,
           COUNT(*) FILTER (WHERE hours_since_prev_phys < 0)           AS temps_negatif,
           COUNT(*) FILTER (WHERE n_purchases_90d = 0)                 AS sans_historique_90j
    FROM '{CTX}'
""").df())

con.sql(f"""
    SELECT ROUND(MEDIAN(dist_home_km), 1)        AS mediane_dist_domicile_km,
           ROUND(MAX(dist_home_km), 1)           AS max_dist_domicile_km,
           ROUND(MEDIAN(speed_kmh), 1)           AS mediane_vitesse_kmh,
           ROUND(MAX(speed_kmh), 0)              AS max_vitesse_kmh,
           ROUND(MEDIAN(amt_ratio_avg_90d), 2)   AS mediane_ratio_montant
    FROM '{CTX}'
""").df().T

In [ ]:
con.sql(f"""
    SELECT t.is_fraud,
           COUNT(*)                                              AS transactions,
           ROUND(MEDIAN(c.amt_ratio_avg_90d), 2)                 AS mediane_ratio_montant,
           ROUND(MEDIAN(c.dist_home_km), 1)                      AS mediane_dist_domicile_km,
           ROUND(100 * AVG(c.country_new::INT), 3)               AS pct_pays_nouveau,
           ROUND(100 * AVG(c.mcc_new::INT), 2)                   AS pct_mcc_nouveau,
           ROUND(100 * AVG(c.merchant_new::INT), 2)              AS pct_commercant_nouveau,
           ROUND(100 * AVG((c.speed_kmh > 900)::INT), 3)         AS pct_vitesse_sup_900kmh
    FROM '{TX}' t
    JOIN '{CTX}' c USING (tx_id)
    WHERE t.split = 'train'
    GROUP BY t.is_fraud
    ORDER BY t.is_fraud
""").df().T

## Conclusion
Features calculées uniquement sur le passé strict (tests anti-fuite : `test_velocity.py`, `test_context.py`).

| Fichier | Lignes | Durée | Contenu |
|---|---|---|---|
| velocity.parquet | 24 386 900 | 40 s | transactions, montants et erreurs sur 1 h / 24 h / 7 j par carte |
| context.parquet | 24 386 900 | 341 s | montant relatif (90 j), distances, vitesse, nouveauté pays/MCC/commerçant |

**Contrôles** : 0 distance impossible (> 20 016 km), 6 139 cartes sans historique,
2 000 clients sans achat physique précédent — tous conformes.

**Pouvoir discriminant (train)** : pays jamais vu × 227, commerçant nouveau × 17,
vitesse > 900 km/h × 16, distance au domicile × 46 (médiane), montant relatif × 2,1.

**Limites** : 1,2 % des transactions légitimes dépassent 900 km/h (artefact du générateur IBM)
→ aucune règle à seuil, combinaison apprise par le modèle. Vitesses extrêmes dues à l'arrondi
à la minute (minimum 1 min).

**Incident** : le premier calcul a saturé la mémoire WSL (7,6 Gio) et fait tomber VS Code Server.
Correction : DuckDB limité à 3 Go / 4 threads, débordement sur le disque Linux.